# 混合精度学習（Automatic Mixed Precision）

---
## 目的
GPU上での学習を高速化・省メモリ化する手法である混合精度学習（Automatic Mixed Precision, AMP）の仕組みを理解する．
また，`torch.autocast`と`torch.amp.GradScaler`を用いた実装方法を身につける．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 混合精度学習とは

これまでのノートブックでは，ネットワークのパラメータや計算はすべて32bit浮動小数点数（`float32`）で扱ってきました．
混合精度学習では，計算の種類に応じて`float32`と16bit浮動小数点数（`float16`）を自動的に使い分けながら学習を行います．`float16`はデータサイズが`float32`の半分であるため，

- GPUメモリの使用量を削減できる
- 近年のGPUに搭載されているTensor Coreなどの演算ユニットにより，`float16`での行列演算を高速に実行できる

といった利点があります．一方で，`float16`は表現できる数値の範囲が`float32`よりも狭いため，勾配の値が非常に小さくなった際に0になってしまう（アンダーフローする）ことがあり，そのままでは学習がうまく進まない場合があります．
そこで，PyTorchでは以下の2つの仕組みを組み合わせて，安全に混合精度学習を行います．

- **`torch.autocast`**：`with`文で囲んだ範囲内の計算について，演算ごとに適した精度（`float16`または`float32`）を自動的に選択して実行するコンテキストマネージャ．
- **`torch.amp.GradScaler`**：誤差（loss）の値を大きくスケールしてから逆伝播することで，勾配のアンダーフローを防ぐための仕組み．`optimizer.step()`の前に，スケールした分を元に戻す（アンスケールする）処理を行う．

## データセットとネットワークの準備
MNISTデータセットと畳み込みニューラルネットワーク（CNN）を用意します．

In [ ]:
train_data = torchvision.datasets.MNIST(root="./", train=True, transform=transforms.ToTensor(), download=True)
test_data = torchvision.datasets.MNIST(root="./", train=False, transform=transforms.ToTensor(), download=True)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
        self.l1 = nn.Linear(14*14*4, 10)
        self.act = nn.Sigmoid()
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        h = self.pool(self.act(self.conv1(x)))
        h = h.view(h.size()[0], -1)
        h = self.l1(h)
        return h

## 通常（float32）での学習

まずは比較のため，これまでと同様に`float32`のみを用いて学習を行い，所要時間と使用したGPUメモリ量を記録します．

In [ ]:
batch_size = 100
epoch_num = 3

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

model = CNN()
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()

train_start = time()
for epoch in range(1, epoch_num+1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)
        loss = criterion(y, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}".format(epoch, sum_loss/len(train_loader), count.item()/len(train_data)))

fp32_time = time() - train_start
fp32_memory = torch.cuda.max_memory_allocated() / 1e6 if device.type == 'cuda' else 0
print("elapsed time: {:.2f} sec, peak GPU memory: {:.1f} MB".format(fp32_time, fp32_memory))

## AMPを用いた学習（float16）

続いて，`torch.autocast`と`torch.amp.GradScaler`を用いて，同じネットワーク構造・同じ設定でAMPによる学習を行います．

学習ループの変更点は次の3つです．

1. 順伝播と誤差の計算を`with torch.autocast(device_type=device.type, dtype=torch.float16):`の範囲内で行う．
2. `loss.backward()`の代わりに，`scaler.scale(loss).backward()`でスケールした誤差を逆伝播する．
3. `optimizer.step()`の代わりに，`scaler.step(optimizer)`でアンスケールしてからパラメータを更新し，その後`scaler.update()`でスケール係数を更新する．

In [ ]:
model_amp = CNN()
model_amp = model_amp.to(device)

optimizer = torch.optim.SGD(model_amp.parameters(), lr=0.01, momentum=0.9)
scaler = torch.amp.GradScaler(device.type)

model_amp.train()

if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()

train_start = time()
for epoch in range(1, epoch_num+1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        with torch.autocast(device_type=device.type, dtype=torch.float16):
            y = model_amp(image)
            loss = criterion(y, label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}".format(epoch, sum_loss/len(train_loader), count.item()/len(train_data)))

amp_time = time() - train_start
amp_memory = torch.cuda.max_memory_allocated() / 1e6 if device.type == 'cuda' else 0
print("elapsed time: {:.2f} sec, peak GPU memory: {:.1f} MB".format(amp_time, amp_memory))

## bfloat16を用いた学習

PyTorchのAMPでは，`float16`のほかに**bfloat16**（Brain Floating Point）と呼ばれる16bit浮動小数点形式も利用できます．
どちらも16bitで数値を表現しますが，符号部・指数部（数値の範囲を決める部分）・仮数部（数値の精度を決める部分）へのビットの割り当てが異なります．

| 形式 | 符号部 | 指数部 | 仮数部 | 特徴 |
| :-: | :-: | :-: | :-: | :-- |
| `float32` | 1bit | 8bit | 23bit | 基準となる精度・数値範囲 |
| `float16` | 1bit | 5bit | 10bit | 精度は`float32`に近いが，表現できる数値の範囲が狭い |
| `bfloat16` | 1bit | 8bit | 7bit | 数値の範囲は`float32`と同じだが，精度（仮数部のビット数）は`float16`より低い |

`bfloat16`は`float32`と同じ指数部のビット数を持つため，表現できる数値の範囲が`float32`と同程度であり，勾配が非常に小さくなってもアンダーフローが起こりにくいという特徴があります．
そのため，`bfloat16`を用いる場合は，多くの場合`GradScaler`を使わずに学習できます．一方で，仮数部のビット数が`float16`より少ないため，数値1つあたりの精度（有効桁数）は`float16`よりも低くなります．

なお，`bfloat16`によるTensor Coreを用いた計算の高速化は，NVIDIAのAmpere世代（RTX 30シリーズやA100など）以降のGPUでサポートされています．それより前の世代のGPUでは，計算自体は可能ですが，高速化の効果は得られない場合があります．

In [ ]:
model_bf16 = CNN()
model_bf16 = model_bf16.to(device)

optimizer = torch.optim.SGD(model_bf16.parameters(), lr=0.01, momentum=0.9)

model_bf16.train()

if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()

train_start = time()
for epoch in range(1, epoch_num+1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            y = model_bf16(image)
            loss = criterion(y, label)

        ### bfloat16はfloat32と同程度の数値範囲を持つため，GradScalerを使わずに逆伝播できる
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}".format(epoch, sum_loss/len(train_loader), count.item()/len(train_data)))

bf16_time = time() - train_start
bf16_memory = torch.cuda.max_memory_allocated() / 1e6 if device.type == 'cuda' else 0
print("elapsed time: {:.2f} sec, peak GPU memory: {:.1f} MB".format(bf16_time, bf16_memory))

## 速度・メモリ使用量・精度の比較

`float32`，AMP（`float16`），AMP（`bfloat16`）の3つについて，所要時間・ピーク時のGPUメモリ使用量・テストデータでの精度を比較します．

In [ ]:
def evaluate(model):
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)
    model.eval()
    count = 0
    with torch.no_grad():
        for image, label in test_loader:
            image = image.to(device)
            label = label.to(device)
            y = model(image)
            pred = torch.argmax(y, dim=1)
            count += torch.sum(pred == label)
    return count.item() / len(test_data)

fp32_acc = evaluate(model)
amp_acc = evaluate(model_amp)
bf16_acc = evaluate(model_bf16)

print("{:<18}{:>12}{:>16}{:>12}".format("", "time [sec]", "peak mem [MB]", "test acc"))
print("{:<18}{:>12.2f}{:>16.1f}{:>12.4f}".format("float32", fp32_time, fp32_memory, fp32_acc))
print("{:<18}{:>12.2f}{:>16.1f}{:>12.4f}".format("AMP (float16)", amp_time, amp_memory, amp_acc))
print("{:<18}{:>12.2f}{:>16.1f}{:>12.4f}".format("AMP (bfloat16)", bf16_time, bf16_memory, bf16_acc))

GPUの世代やネットワークの規模，バッチサイズによって効果の大きさは異なりますが，AMPを用いることでGPUメモリ使用量を削減しつつ，同程度の精度を維持できていることが確認できます．
今回使用したネットワークは非常に小さいため速度差は限定的ですが，Transformerなどの大規模なネットワークではAMPによる速度向上効果がより顕著になります．

`float16`と`bfloat16`のどちらを使うべきか迷った場合は，まず`GradScaler`が不要で実装がより簡単な`bfloat16`を試し，対応していないGPU（Ampere世代より前）を使用する場合や，`bfloat16`では精度が不足する場合に`float16`（+`GradScaler`）を検討するとよいでしょう．

なお，GPUが利用できない環境（CPUのみ）では，AMPによる高速化の効果は得られません．「ランタイム→ランタイムのタイプを変更」からGPUを選択してください．

## 課題

1. ネットワークをより大きな構造に変更したり，バッチサイズを大きくしたりして，AMPによる速度・メモリ削減の効果がどのように変化するか確認してみましょう．